# GPT reproduction first model

In [2]:
import torch

In [3]:
# 1. Read the input file.
import numpy as np

def generate_training_data_standard_distribution(size:int) -> str:
    data_int = np.random.randint(ord('a'),ord('z') + 1,size=size).tolist()
    return "".join([chr(x) for x in data_int])

def generate_training_data_special_pattern(size:int) -> str:
    data_int = []
    i = 0
    while i < size-1:
        item = int(np.random.randint(ord('a'),ord('z') + 1))
        data_int.append(item)
        i += 1
        if item == ord('a'):
            i += 1
            random2 = np.random.rand()
            if random2 <= 0.5:
                data_int.append(ord("b"))
            elif random2 > 0.5 and random2 <= 0.7:
                data_int.append(ord("c"))
            elif random2 > 0.7 and random2 <= 0.8:
                data_int.append(ord("d"))
            else:
                data_int.append(int(np.random.randint(ord('e'),ord('z') + 1)))
    if i != size:
        data_int.append(int(np.random.randint(ord('b'),ord('z') + 1)))
    return "".join([chr(x) for x in data_int])

generate_training_data = generate_training_data_special_pattern

# 2. Create the character-level vocabulary.

class Vocabulary:
    def __init__(self,data):
        self.vocabulary = sorted(list(set(list(training_data))))
        self.stoi = { c:i for i,c in enumerate(self.vocabulary) }
        self.itos = { i:c for i,c in enumerate(self.vocabulary) }

# 3. Create decode and encode function
    def encode_data(self,input:str) -> list:
        return [self.stoi[x] for x in input ]

    def decode_data(self,input:list) -> str:
        return "".join([self.itos[x] for x in input ])

    def size(self) -> int:
        return len(self.vocabulary)

training_data = generate_training_data(100000)

print("a",training_data.count("a"))
print("ab",training_data.count("ab"))
print("ac",training_data.count("ac"))
print("ad",training_data.count("ad"))
print("ad",training_data.count("ae"))
vocabulary = Vocabulary(training_data)
print(vocabulary.encode_data("bank"))
print(vocabulary.decode_data(vocabulary.encode_data("bank")))
# ----- PARAMETERS -----
EMBEDDING_DIM = vocabulary.size()


a 3539
ab 1766
ac 668
ad 358
ad 37
[1, 0, 13, 10]
bank


In [4]:
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 2 # what is the maximum context length for predictions?

Ez a model most reprodukálja az lehetö legegyszerübb nyelvi modelt. Csak a loss kiszámítását tartalmazza, és a következö token generátor funkcióját

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F


class LLM_Forward_Loss_Generator(nn.Module):  # What is the torch.nn.Module?

    def __init__(self, vocabulary: Vocabulary, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.token_embedding_table = nn.Embedding(vocabulary.size(), EMBEDDING_DIM)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx)  # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss


torch.manual_seed(1337)

network = LLM_Forward_Loss_Generator(vocabulary)
input = torch.tensor(
    [vocabulary.encode_data("al"), vocabulary.encode_data("ko")], dtype=torch.long
)
target = torch.tensor(
    [vocabulary.encode_data("al"), vocabulary.encode_data("ko")], dtype=torch.long
)
print(network.token_embedding_table(input).shape)

prediction, loss = network(input, target)
print(prediction, loss)

torch.Size([2, 2, 26])
tensor([[ 0.1808, -0.0700, -0.3596, -0.9152,  0.6258,  0.0255,  0.9545,  0.0643,
          0.3612,  1.1679, -1.3499, -0.5102,  0.2360, -0.2398, -0.9211,  1.5433,
          1.3488, -0.1396,  0.2858,  0.9651, -2.0371,  0.4931,  1.4870,  0.5910,
          0.1260, -1.5627],
        [-0.6576, -2.5729,  0.0210,  1.0060, -1.2492,  0.2441, -0.6387, -0.3186,
         -1.2942, -1.0726,  0.2290, -0.9001,  0.6614,  0.5118,  0.6762, -1.3639,
          0.5486,  0.0895,  0.3575, -1.6521, -0.7584,  0.0695,  0.9937, -0.2821,
          1.1088, -1.9881],
        [ 0.8148, -0.0643,  1.4237,  0.2617, -1.8528,  0.2019, -1.1787, -0.1036,
         -1.7830, -0.8323, -0.4346, -1.2480, -0.2880,  0.8809, -0.7190,  0.1745,
          0.7520, -0.0629, -0.7111,  0.9810, -0.7244, -1.5010, -2.8348, -2.8272,
         -0.1736,  0.0512],
        [ 1.6685,  0.5976, -1.8736,  1.2910, -0.3753, -1.8943,  0.5557,  0.8567,
         -0.8461,  0.5015, -0.9656, -0.7255,  0.0990,  0.5928, -0.0422, -0.9566,
  